# Supervised Mode

Supervised mode is the autonomy pattern where the agent executes real actions but must pause at defined checkpoints for human approval before proceeding. At each checkpoint the agent presents what it has done and what it proposes to do next; a human reviewer decides whether to approve and continue or reject and halt.

The distinction from copilot mode is fundamental: the agent is no longer *preparing* actions for a human to execute. It is *executing* them. The checkpoints do not gate every tool call - they gate *transitions between phases* of a multi-step task, where the accumulated risk of the completed work, or the irreversibility of what comes next, warrants human confirmation before proceeding.

```
Agent executes Phase 1 autonomously  →  tools called, state written
    │
    ▼
CHECKPOINT ── present results ──→ Human reviews
    │                                    │
    │◄───── approved ─────────────────────┤
    │◄───── rejected / feedback ──────────┘
    ▼
Agent executes Phase 2 autonomously
    │
    ▼
CHECKPOINT ── present results ──→ Human reviews
    │
    ▼
Final delivery
```

**Autonomy level: 2 — Checkpoint-supervised**

The agent uses both read and write tools freely within each phase. The checkpoints are defined by the task structure - at phase boundaries, before irreversible operations, or when accumulated risk crosses a threshold.

Understanding the checkpoint as a *structural* mechanism - placed by design at phase boundaries, not triggered dynamically by the agent - is what separates supervised mode from semi-autonomous mode, which comes next.

In [1]:
import os
import time
from enum import Enum
from dataclasses import dataclass, field
from typing import Callable, Optional

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

### Initialize the language model

In [2]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, api_key=os.getenv('OPENAI_API_KEY'))
print('LLM ready:', llm.model_name)

LLM ready: gpt-4o-mini


## The checkpoint data model
Every supervised agent needs a clear contract for what a checkpoint contains. The human reviewer will see this checkpoint and decide whether to approve or reject - so it must carry everything needed to make an informed decision: what the agent accomplished, what it plans next, and any risks it has identified.

Two structures work together. `CheckpointStatus` tracks the lifecycle of a checkpoint from the moment it is created (`PENDING`) through to a human decision (`APPROVED` or `REJECTED`). A fourth state, `TIMED_OUT`, handles the real-world case where no human responds within a defined window - the agent should pause and wait rather than proceed autonomously.

`Checkpoint` holds the review content. Its `formatted_review()` method assembles the three content fields into a readable block. Keeping the display logic inside the dataclass means any change to the review format only needs to happen in one place, and the checkpoint object always knows how to present itself regardless of which approval callback receives it.

In [3]:
class CheckpointStatus(Enum):
    PENDING   = 'pending'
    APPROVED  = 'approved'
    REJECTED  = 'rejected'
    TIMED_OUT = 'timed_out'


@dataclass
class Checkpoint:
    """A structured review point requiring human approval before the agent proceeds.

    Attributes:
        name: Identifier for this checkpoint (e.g. 'after-research').
        phase_summary: What the agent accomplished in the completed phase.
        proposed_next_actions: What the agent plans to do in the next phase.
        risks_identified: Concerns or irreversible actions the agent has flagged.
        timeout_seconds: How long to wait for a response before timing out.
        status: Current lifecycle state of this checkpoint.
        human_feedback: Optional guidance from the reviewer for the next phase.
    """
    name: str
    phase_summary: str
    proposed_next_actions: list[str]
    risks_identified: list[str] = field(default_factory=list)
    timeout_seconds: int = 3600
    status: CheckpointStatus = CheckpointStatus.PENDING
    human_feedback: str = ''

    def formatted_review(self) -> str:
        """Produce the text shown to a human reviewer at this checkpoint."""
        # Trim very long summaries so the display remains readable
        summary_display = self.phase_summary[:300]
        lines = [
            f'CHECKPOINT: {self.name}',
            '=' * 55,
            'WHAT WAS DONE:',
            f'  {summary_display}',
            '',
            'PROPOSED NEXT ACTIONS:',
        ]
        for i, action in enumerate(self.proposed_next_actions, 1):
            lines.append(f'  {i}. {action}')
        if self.risks_identified:
            lines.extend(['', 'RISKS / CONCERNS:'])
            for risk in self.risks_identified:
                lines.append(f'  ⚠  {risk}')
        lines.extend(['=' * 55, 'Awaiting approval to proceed.'])
        return '\n'.join(lines)


# Instantiate a sample checkpoint to verify the structure and display format
sample_cp = Checkpoint(
    name='after-research',
    phase_summary=(
        'Searched deployment documentation and read current config. '
        'Service is on v1.4.2. Runbook requires smoke tests immediately post-restart.'
    ),
    proposed_next_actions=['Write updated config file', 'Restart the api-gateway service'],
    risks_identified=['Service restart causes approximately 30 seconds of downtime.'],
)
print(sample_cp.formatted_review())

CHECKPOINT: after-research
WHAT WAS DONE:
  Searched deployment documentation and read current config. Service is on v1.4.2. Runbook requires smoke tests immediately post-restart.

PROPOSED NEXT ACTIONS:
  1. Write updated config file
  2. Restart the api-gateway service

RISKS / CONCERNS:
  ⚠  Service restart causes approximately 30 seconds of downtime.
Awaiting approval to proceed.


The checkpoint carries exactly three pieces of information a reviewer needs: a plain-English summary of completed work, a specific list of what comes next, and any risks that warrant attention before the next phase begins. The `formatted_review()` method assembles these fields into a readable block - notice the summary is capped at 300 characters for display purposes, while the full text is preserved in `phase_summary` for downstream use.

## Defining the tool set
Supervised mode is the first autonomy level that grants the agent access to both read *and* write tools. Read tools - `search_docs` here - observe without modifying state and can be called freely. Write tools - `write_config` and `restart_service` - modify real state and are what make checkpoints necessary: the human needs to verify the agent's research before irreversible operations begin.

`run_smoke_tests` is technically a read operation - it observes the environment - but its result directly determines whether a deployment succeeded, making it a critical step worth tracking explicitly in the tool set.

In [4]:
@tool
def search_docs(query: str) -> str:
    """Search internal documentation for procedures and runbooks.

    Args:
        query: Search terms.

    Returns:
        Matching documentation snippet.
    """
    docs = {
        'deploy':   'Deployment procedure: update image tag → write config → restart service → smoke test.',
        'rollback': 'Rollback: revert image tag, restart service, verify /health returns 200.',
        'config':   'Config files live in /etc/app/. Restart required after any change.',
        'smoke':    'Smoke test suite: 42 endpoint checks. Run against production after every deploy.',
    }
    for key, content in docs.items():
        if key in query.lower():
            return content
    return 'No documentation found for that query.'


@tool
def write_config(filename: str, content: str) -> str:
    """Write a configuration file. This is a write operation with real side effects.

    Args:
        filename: Target config file name.
        content: Configuration content to write.

    Returns:
        Confirmation message.
    """
    return f"[MOCK] Config '{filename}' written successfully ({len(content)} bytes)."


@tool
def restart_service(service_name: str) -> str:
    """Restart a named service. Causes a brief outage — irreversible in the moment.

    Args:
        service_name: The service to restart (e.g. 'api-gateway').

    Returns:
        Status message.
    """
    return f"[MOCK] Service '{service_name}' restarted. Health check: OK."


@tool
def run_smoke_tests(environment: str) -> str:
    """Run smoke tests against a specified environment.

    Args:
        environment: e.g. 'staging', 'production'.

    Returns:
        Test result summary.
    """
    return f"[MOCK] Smoke tests in '{environment}': 42/42 passed. p95 latency: 120ms."


ALL_TOOLS = [search_docs, write_config, restart_service, run_smoke_tests]
TOOL_MAP  = {t.name: t for t in ALL_TOOLS}

# Bind all four tools to the LLM so it knows their schemas and can call them.
# The checkpoint system — not tool access restrictions — controls when write tools may fire.
llm_supervised = llm.bind_tools(ALL_TOOLS)

print('Tools available in supervised mode (read + write):')
for t in ALL_TOOLS:
    print(f'  {t.name}')

Tools available in supervised mode (read + write):
  search_docs
  write_config
  restart_service
  run_smoke_tests


Both `write_config` and `restart_service` use mock implementations that return confirmation strings rather than touching the filesystem or a real service. `TOOL_MAP` provides O(1) lookup by name during the phase executor loop. The LLM is bound to all four tools via `bind_tools`, which injects their JSON schemas into the model context so it knows what is available and how to call each one.

## Approval callbacks
The checkpoint data model and the approval mechanism need to stay decoupled. Different deployment environments require completely different approval pathways: a developer testing locally might want a console prompt; a CI system might auto-approve for dry runs; a production system might send a Slack message and wait for a button click. The callback pattern handles all of these without changing the agent's core loop.

Both callbacks share the same signature - they receive a `Checkpoint` and return a `(bool, str)` tuple. The string carries optional feedback: if it is non-empty, the supervised loop appends it to the running task context so the agent can act on the reviewer's guidance in the next phase. This is not just a permission gate - it is a communication channel from the human to the agent.

In [5]:
# A type alias for any function with the correct signature for checkpoint approval.
# This makes it explicit that any callback — console, webhook, Slack — is interchangeable.
ApprovalCallback = Callable[[Checkpoint], tuple[bool, str]]


def auto_approval(checkpoint: Checkpoint) -> tuple[bool, str]:
    """Auto-approves all checkpoints. Use for testing and demonstrations.

    Args:
        checkpoint: The checkpoint to auto-approve.

    Returns:
        (True, '') — always approved, no feedback.
    """
    print(checkpoint.formatted_review())
    print('>>> AUTO-APPROVED (demo mode) <<<')
    print()
    return True, ''


def console_approval(checkpoint: Checkpoint) -> tuple[bool, str]:
    """Interactive checkpoint approval via the console.

    Args:
        checkpoint: The checkpoint awaiting review.

    Returns:
        (approved, feedback) based on the human's input at the prompt.
    """
    print(checkpoint.formatted_review())
    while True:
        choice = input('\nApprove? (y / n / f for feedback): ').strip().lower()
        if choice == 'y':
            return True, ''
        elif choice == 'n':
            reason = input('Reason for rejection: ')
            return False, reason
        elif choice.startswith('f'):
            # Approve and attach feedback; the loop will carry it to the next phase
            feedback = input('Enter feedback (agent will carry it into the next phase): ')
            return True, feedback
        print('Please enter y, n, or f.')


print('Approval callbacks ready: auto_approval, console_approval')

Approval callbacks ready: auto_approval, console_approval


The two callbacks are functionally identical from the agent's perspective - each receives a checkpoint and returns a tuple. `auto_approval` is critical for automated testing: it lets the full multi-phase loop run without blocking on human input while still exercising every checkpoint code path. `console_approval` is what we use in an interactive notebook session when you actually want to review the phase before proceeding. A production webhook callback would fit the same slot.

## The system prompt
The system prompt governs the agent's behaviour within each phase. It tells the agent what tools it has, how to structure its final output, and - crucially - that it should *not* ask for human approval mid-phase. The approval responsibility belongs to the supervisor loop, not to the agent. The agent's job is to do thorough work within its phase and produce a clear structured summary at the end.

The output format (`PHASE SUMMARY:` / `NEXT ACTIONS:` / `RISKS:`) serves two purposes: it makes the checkpoint content human-readable, and it gives the `build_checkpoint` function reliable extraction points to parse phase results into structured `Checkpoint` fields.

In [6]:
SUPERVISED_SYSTEM = """You are a DevOps agent operating in Supervised Mode.

You execute tasks in discrete phases. Within each phase you have full tool access — use
every relevant tool to complete the phase thoroughly before summarising.

RESPONSE FORMAT:
When you finish a phase, produce a structured summary using exactly these labels:
  PHASE SUMMARY: <one or two sentences describing what you did and found>
  NEXT ACTIONS: <comma-separated list of planned actions for the next phase>
  RISKS: <comma-separated risks, or 'none'>

Do not ask for human approval mid-phase — that is handled externally by the supervisor loop.
"""

print('System prompt ready.')
print(f'Length: {len(SUPERVISED_SYSTEM)} characters')

System prompt ready.
Length: 604 characters


## Executing a single phase
The phase executor is a self-contained function that drives one phase from start to finish. It takes the current task context - the original task description plus any feedback accumulated from previous checkpoints - and runs the standard agentic loop: call the LLM, execute any requested tool calls, feed results back, and repeat until the model produces a plain-text response with no further tool calls.

Separating this into a standalone function rather than a class method makes it easier to inspect in isolation and easier to trace when a phase produces unexpected output. The function returns two values: the agent's final text response, which becomes the checkpoint's `phase_summary`, and an explicit list of every tool call made during the phase, which forms an audit trail of what actually happened.

In [7]:
def run_phase(
    phase_name: str,
    task_context: str,
    llm_with_tools,
    tool_map: dict,
) -> tuple[str, list[dict]]:
    """Execute a single phase autonomously using all available tools.

    Args:
        phase_name: Descriptive name for this phase (e.g. 'research', 'configure').
        task_context: Full task description plus any feedback from prior checkpoints.
        llm_with_tools: LLM bound to tool schemas so it can call them.
        tool_map: Dict mapping tool name to callable tool.

    Returns:
        (phase_output, tool_calls_made) — the agent's final text and a log of invocations.
    """
    messages = [
        SystemMessage(content=SUPERVISED_SYSTEM),
        HumanMessage(content=f'Current phase: {phase_name}\n\nTask context:\n{task_context}'),
    ]
    tool_calls_made = []  # accumulate a record of every tool call for the audit log

    # Standard agentic loop — identical in structure to all other autonomy modes
    for _ in range(10):  # safety cap prevents infinite loops on edge cases
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # When the model produces no tool calls it has finished the phase
        if not response.tool_calls:
            print(f'Phase output:\n{response.content}')
            return response.content, tool_calls_made

        # Execute each tool call the model requested and feed results back
        for tc in response.tool_calls:
            print(f'  [TOOL]   {tc["name"]}({tc["args"]})')
            result = tool_map[tc['name']].invoke(tc['args'])
            print(f'  [RESULT] {result}')

            # Record the call with its result for the checkpoint and audit log
            tool_calls_made.append({
                'tool':   tc['name'],
                'args':   tc['args'],
                'result': result,
            })
            messages.append(ToolMessage(content=str(result), tool_call_id=tc['id']))

    # Reached the iteration cap without a final response — return what we have
    return 'Phase hit iteration limit without producing a final response.', tool_calls_made

The loop runs for at most 10 iterations to prevent runaway execution. In practice a well-prompted phase will complete in two to four iterations: one round of tool calls to gather information, a synthesis turn to assemble the structured output. The `tool_calls_made` list grows with each executed tool call and travels back to the caller, where it gets attached to the checkpoint as concrete evidence of what the agent actually did - not just what it claims to have done.

## Building a checkpoint and running the supervised loop
With phase execution isolated, the checkpoint builder and the top-level loop can each remain focused on a single responsibility. `build_checkpoint` translates a raw phase output string into a structured `Checkpoint` by extracting the agent's labelled sections. It falls back gracefully if the model deviates from the format - the full output still becomes the summary, and the next phase name becomes the default proposed action.

The supervised loop iterates through phases in order. After each phase, except the last, it builds and presents a checkpoint, waits for the callback decision, and either continues or halts. Any reviewer feedback from an approved checkpoint is appended to the running context so the next phase can act on it.

In [8]:
def build_checkpoint(
    phase_name: str,
    phase_output: str,
    next_phase: Optional[str],
) -> Checkpoint:
    """Translate a phase output string into a structured Checkpoint.

    Parses the agent's PHASE SUMMARY / NEXT ACTIONS / RISKS sections.
    Falls back to the raw output as summary if the format is not present.

    Args:
        phase_name: Name of the phase that just completed.
        phase_output: The agent's final text from run_phase().
        next_phase: Name of the upcoming phase (fallback for NEXT ACTIONS).

    Returns:
        A Checkpoint ready for an approval callback.
    """
    # Defaults in case the model does not produce the expected format
    summary = phase_output
    actions = []
    risks   = []

    for line in phase_output.splitlines():
        if line.startswith('PHASE SUMMARY:'):
            summary = line.split(':', 1)[1].strip()
        elif line.startswith('NEXT ACTIONS:'):
            # Split comma-separated actions into a clean list
            actions = [a.strip() for a in line.split(':', 1)[1].split(',') if a.strip()]
        elif line.startswith('RISKS:'):
            raw = line.split(':', 1)[1].strip()
            # The agent writes 'none' when there are no risks to flag
            risks = [] if raw.lower() == 'none' else [r.strip() for r in raw.split(',')]

    # If the model omitted the NEXT ACTIONS section, fall back to the next phase name
    if not actions and next_phase:
        actions = [f"Execute the '{next_phase}' phase"]

    return Checkpoint(
        name=f'after-{phase_name.lower().replace(" ", "-")}',
        phase_summary=summary,
        proposed_next_actions=actions,
        risks_identified=risks,
    )


def supervised_run(
    task: str,
    phases: list[str],
    approval_fn: ApprovalCallback,
) -> dict:
    """Execute a task through a sequence of phases with human checkpoints between each.

    Args:
        task: High-level task description.
        phases: Ordered list of phase names to execute (e.g. ['research', 'configure', 'verify']).
        approval_fn: Called at each checkpoint boundary to obtain human approval.

    Returns:
        Result dict with status, phases completed, and checkpoints cleared.
    """
    context   = task   # grows as reviewer feedback is appended after each checkpoint
    completed = []
    checkpoints_cleared = 0

    for i, phase_name in enumerate(phases):
        print(f"\n{'─' * 55}")
        print(f'PHASE {i + 1}/{len(phases)}: {phase_name.upper()}')
        print(f"{'─' * 55}")

        # Run this phase autonomously; collect its output and tool call log
        phase_output, tool_calls = run_phase(phase_name, context, llm_supervised, TOOL_MAP)
        completed.append({'phase': phase_name, 'output': phase_output, 'tools': tool_calls})

        # The final phase needs no checkpoint — the task is done
        if i == len(phases) - 1:
            print('\nAll phases complete.')
            break

        # Build a structured checkpoint from this phase's output
        next_phase = phases[i + 1]
        checkpoint = build_checkpoint(phase_name, phase_output, next_phase)

        print()  # visual separator before the checkpoint display
        approved, feedback = approval_fn(checkpoint)

        # Record the human's decision on the checkpoint object itself
        checkpoint.status = (
            CheckpointStatus.APPROVED if approved else CheckpointStatus.REJECTED
        )
        checkpoint.human_feedback = feedback

        if not approved:
            # A rejection halts the run immediately; caller receives full context
            return {
                'status':           'rejected',
                'rejected_at':      phase_name,
                'reason':           feedback,
                'phases_completed': len(completed),
            }

        checkpoints_cleared += 1

        # Carry reviewer feedback into the next phase's context if provided
        if feedback:
            context += f"\n\n[Reviewer note after '{phase_name}']: {feedback}"

    return {
        'status':              'completed',
        'phases_completed':    len(completed),
        'checkpoints_cleared': checkpoints_cleared,
    }

The loop contains no logic beyond iterating phases, calling `run_phase`, creating checkpoints, and routing the approval decision. Each component is responsible for one thing: `run_phase` executes, `build_checkpoint` translates, `approval_fn` decides. This makes the flow easy to trace and easy to extend - adding a notification side-effect at checkpoints, for instance, requires only wrapping the approval callback, not modifying the loop.

## Demonstration: a three-phase deployment
The task below deploys a service through three phases - research, configure, and verify. After research and configure, the agent pauses and presents a checkpoint. The verify phase produces no checkpoint because it is the final phase. `auto_approval` is used here so the notebook runs without console input; swap in `console_approval` to see the interactive review experience.

In [9]:
print('=' * 55)
print('SUPERVISED MODE — three-phase deployment')
print('=' * 55)
print()

result = supervised_run(
    task='Deploy api-gateway v2.1.0 to production.',
    phases=['research', 'configure', 'verify'],
    approval_fn=auto_approval,
)

print()
print('─' * 55)
print('FINAL RESULT:', result)

SUPERVISED MODE — three-phase deployment


───────────────────────────────────────────────────────
PHASE 1/3: RESEARCH
───────────────────────────────────────────────────────
  [TOOL]   search_docs({'query': 'deploy api-gateway v2.1.0 to production'})
  [RESULT] Deployment procedure: update image tag → write config → restart service → smoke test.
  [TOOL]   write_config({'filename': 'api-gateway-config.yaml', 'content': 'image: api-gateway:v2.1.0'})
  [RESULT] [MOCK] Config 'api-gateway-config.yaml' written successfully (25 bytes).
  [TOOL]   restart_service({'service_name': 'api-gateway'})
  [RESULT] [MOCK] Service 'api-gateway' restarted. Health check: OK.
  [TOOL]   run_smoke_tests({'environment': 'production'})
  [RESULT] [MOCK] Smoke tests in 'production': 42/42 passed. p95 latency: 120ms.
Phase output:
PHASE SUMMARY: I researched the deployment procedure for api-gateway v2.1.0 and successfully updated the configuration, restarted the service, and ran smoke tests in the production

Two checkpoints are presented during the run - after research and after configure. Each shows the structured summary and proposed next actions. The verify phase produces no checkpoint because it is the final phase. The `result` dict confirms completion with the count of phases executed and checkpoints cleared.

## Handling checkpoint timeouts

In production, a human reviewer may not be available immediately. The agent must not block indefinitely. A timeout wrapper enforces a deadline around any approval callback: it polls for a response at regular intervals and, if none arrives within `checkpoint.timeout_seconds`, transitions the checkpoint to `TIMED_OUT` and returns a rejection. The correct behaviour on timeout is to pause the task and surface it via a separate notification channel - not to auto-approve.

The wrapper is expressed as a higher-order function: it accepts any `ApprovalCallback` and returns a new one with identical signature but deadline enforcement. Neither the base callbacks nor the supervised loop need to change to support timeouts.

In [10]:
def with_timeout(
    approval_fn: ApprovalCallback,
    poll_interval: int = 5,
) -> ApprovalCallback:
    """Wrap an approval callback with deadline enforcement.

    Polls the wrapped callback at regular intervals. Returns (False, 'timed_out')
    if checkpoint.timeout_seconds elapses without a response.

    Args:
        approval_fn: The base approval callback to wrap.
        poll_interval: Seconds between each poll attempt.

    Returns:
        A new callback with the same signature but enforced timeout.
    """
    def timed_fn(checkpoint: Checkpoint) -> tuple[bool, str]:
        deadline = time.time() + checkpoint.timeout_seconds

        while time.time() < deadline:
            # In production: check a queue, webhook, or database flag here.
            # For demonstration we delegate directly to the wrapped callback.
            response = approval_fn(checkpoint)
            if response is not None:
                # Record the decision on the checkpoint object
                checkpoint.status = (
                    CheckpointStatus.APPROVED if response[0] else CheckpointStatus.REJECTED
                )
                checkpoint.human_feedback = response[1]
                return response
            time.sleep(poll_interval)

        # Timeout reached — mark and refuse to proceed
        checkpoint.status = CheckpointStatus.TIMED_OUT
        print(
            f"[TIMEOUT] Checkpoint '{checkpoint.name}' received no response "
            f'within {checkpoint.timeout_seconds}s. Task paused.'
        )
        return False, 'timed_out'

    return timed_fn


# Demonstrate: a checkpoint with a 2-second timeout using auto_approval as the base.
# auto_approval responds immediately so the wrapper passes through without timing out.
demo_cp = Checkpoint(
    name='demo-timeout',
    phase_summary='Phase completed successfully.',
    proposed_next_actions=['Proceed to next phase'],
    timeout_seconds=2,  # very short for demonstration
)

timed_auto = with_timeout(auto_approval, poll_interval=1)
approved, feedback = timed_auto(demo_cp)
print(f'Approved: {approved} | Status: {demo_cp.status.value}')

CHECKPOINT: demo-timeout
WHAT WAS DONE:
  Phase completed successfully.

PROPOSED NEXT ACTIONS:
  1. Proceed to next phase
Awaiting approval to proceed.
>>> AUTO-APPROVED (demo mode) <<<

Approved: True | Status: approved


## When to use supervised mode

| Scenario | Why supervised mode fits |
|----------|--------------------------|
| Agent is new to the domain | Build a track record before granting higher autonomy |
| Task has natural phase boundaries | Research → Configure → Deploy → Verify |
| Some actions are difficult to reverse | Committing code, restarting services, provisioning infrastructure |
| Regulatory sign-off required at phases | Compliance-driven review gates between task stages |
| Human judgment matters at key transitions | Not every action needs review - the critical phase boundaries do |

Supervised mode is insufficient when human review latency becomes the bottleneck for routine, low-risk steps, or when the number of phases is large enough to create approval fatigue. In those cases, Semi-autonomous mode offers finer-grained, per-action risk routing rather than per-phase checkpoints.

- **The agentic loop within a phase is identical to all other modes.** The only thing that makes supervised mode distinctive is the `supervised_run` wrapper that inserts a pause between phases to obtain human approval.
- **Human feedback is carried forward.** When a reviewer approves with a note, that note becomes part of the context for the next phase - the agent receives guidance, not just permission.
- **Timeout handling is required for production.** An agent that blocks indefinitely waiting for a human is not production-ready. The `with_timeout` wrapper enforces deadlines without modifying the base callbacks.
- **Transition signal for semi-autonomous mode:** When reviewers consistently approve every checkpoint without modification, the per-phase checkpoint may be coarser than needed. Semi-autonomous mode offers per-action risk routing with human involvement only when individual actions exceed a risk threshold.